In [ ]:
# cnn: 이미지 데이터를 잘 학습하는 신경망
# 이미지 데이터(pixel의 집합)
# - 기본 단위: pixel
#   - rgb는 3가지의 channel
#   - 흑백은 1가지의 channel

from PIL import Image
import numpy as np

img = Image.open("Lenna.png")        # 1. 열기
img_resized = img.resize((300, 300))       # 2. 리사이즈 (원본 안 바뀜, 새 객체 반환)
np_img = np.array(img_resized)              # 3. numpy 배열로 변환

# 주의: PIL의 img.load()[x,y]와 numpy의 np_img[y,x]는 좌표 순서가 반대

<class 'PIL.Image.Image'>


In [ ]:
# ===== 블록 2: 대용량 데이터셋 로딩 (폴더 구조 기반) =====
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# datagen = ImageDataGenerator(rescale=1./255)   # 0~255 픽셀을 0~1로 정규화

# dataset = datagen.flow_from_directory(
#     "dataset/train",           # 폴더 구조: dataset/train/클래스명/이미지들
#     target_size=(180, 180),
#     batch_size=32,
#     class_mode='categorical'
# )
# 폴더 이름 자체가 자동으로 클래스 라벨이 됨

### Convolution
- 필터를 이미지 위에서 슬라이딩시키며 국소 영역을 곱-합 연산으로 압축하는 것,
  이걸 통해 "이 위치에 이런 패턴(모서리, 곡선 등)이 있다"는 정보를 뽑아내는 게 CNN의 Conv2D 층이 하는 일

- **padding을 쓰면** → convolution을 여러 번 반복해도 이미지 크기가 줄어들지 않는다
  - VALID: padding 없음, 층을 거칠 때마다 크기가 계속 줄어듦
  - SAME: 가장자리에 0을 채워서 입력과 출력 크기를 동일하게 유지

In [1]:
"""
=====================================================
CNN 실무 예제 — Conv2D/MaxPool/BatchNorm 블록 구조
=====================================================
"""
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam

model = Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), padding="same", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Block 2 (필터 개수 2배로: 이미지가 작아지는 대신 채널 늘림)
    layers.Conv2D(64, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # 분류부 (Dense = MLP, CNN 전용 아님)
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])

model.summary()

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# 학습 예시:
# model.fit(train_X, train_y, epochs=10, batch_size=64,
#           validation_split=0.2, shuffle=True, verbose=2)

c:\dev\workspace\next_ai\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 592,042 (2.26 MB)

 Trainable params: 591,658 (2.26 MB)

 Non-trainable params: 384 (1.50 KB)

In [2]:
"""
=====================================================
ResNet 실무 예제 — Skip Connection (기울기 소실 완화)
=====================================================
"""
import tensorflow as tf
from tensorflow.keras import layers, Sequential

class ResidualBlock(tf.keras.Model):
    def __init__(self, num_kernels, kernel_size=(3, 3)):
        super().__init__()
        self.conv1 = layers.Conv2D(num_kernels, kernel_size, padding="same", activation="relu")
        self.conv2 = layers.Conv2D(num_kernels, kernel_size, padding="same")
        self.add = layers.Add()
        self.relu = layers.Activation("relu")

    def call(self, x):
        shortcut = x
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.add([out, shortcut])
        return self.relu(out)

model = Sequential([
    layers.Conv2D(64, (3, 3), padding="same", activation="relu", input_shape=(32, 32, 3)),
    layers.MaxPooling2D(2),

    ResidualBlock(64),
    ResidualBlock(64),
    ResidualBlock(64),

    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation="softmax"),
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block (ResidualBlock)  │ (None, 16, 16, 64)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_1                │ (None, 16, 16, 64)     │        73,856 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_2                │ (None, 16, 16, 64)     │        73,856 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,010 (875.04 KB)

 Trainable params: 224,010 (875.04 KB)

 Non-trainable params: 0 (0.00 B)

### RNN 챕터 정리

- **순차 데이터 종류**: 자연어, 시계열

- **SimpleRNN을 쓰면** → hidden state를 다음 시점 계산에 재사용해서 순서 있는 데이터를 처리할 수 있다

- **원-핫 인코딩으로 단어를 표현하면** → 단어 수가 늘어날수록 벡터가 계속 커지고, 대부분 0으로 낭비된다 (실제로 확인: "I" 한 단어에 70차원, 69개가 0)

- **Embedding 레이어를 쓰면** → 원-핫보다 훨씬 작은 차원(예: 5, 100차원)의 학습되는 벡터로 단어를 표현할 수 있다

- **window size**: RNN이 한 번 학습(예측)할 때 보는 과거 데이터 개수. window_size=4면 "최근 4개 값을 보고 다음 1개 값을 예측"

- **SimpleRNN을 여러 층 쌓으면(Deep RNN)** → 표현력은 늘지만, 무조건 성능이 좋아지는 건 아니다 (CNN에서 층을 무작정 늘린다고 좋은 게 아니었던 것과 같은 원리)

- **return_sequences=True를 쓰면** → RNN이 마지막 결과 하나만 넘기지 않고, 모든 타임스텝의 결과를 다음 RNN층에 넘겨준다 (RNN을 여러 층 쌓을 때 필수)

- **return_state=True를 쓰면** → RNN의 최종 hidden state를 별도로 꺼낼 수 있다 (Encoder-Decoder 구조에서 사용)

- **Encoder-Decoder 구조**: 입력 시퀀스를 Encoder가 hidden state 하나로 압축하고, 그 hidden state를 Decoder의 시작점으로 넘겨 새로운 시퀀스를 만들어냄. 번역, 다중 스텝 시계열 예측 등에 사용

In [ ]:
"""
=====================================================
RNN 실무 예제 1 — 기본 RNN (텍스트 임베딩 + 시계열)
=====================================================
"""
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# 텍스트 버전 (Embedding + SimpleRNN + 다중분류)
text_model = Sequential([
    layers.Embedding(input_dim=256, output_dim=100),
    layers.SimpleRNN(20),
    layers.Dense(10, activation="softmax"),
])

# 시계열 버전 (window_size개의 과거값 → 다음 값 예측, 회귀)
window_size = 4
timeseries_model = Sequential([
    layers.SimpleRNN(4, input_shape=(window_size, 1)),
    layers.Dense(1),  # activation 없음 = 회귀
])
timeseries_model.compile(optimizer="adam", loss="mse")

In [ ]:
"""
=====================================================
RNN 실무 예제 2 — Deep RNN (여러 층 쌓기)
=====================================================
"""
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam

window_size = 50

deep_rnn_model = Sequential([
    layers.SimpleRNN(20, return_sequences=True, input_shape=(window_size, 1)),  # 다음 RNN층에 전체 시퀀스 전달
    layers.SimpleRNN(20),  # 마지막 층은 최종 결과만
    layers.Dense(1),
])
deep_rnn_model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")

In [ ]:
"""
=====================================================
RNN 실무 예제 3 — LSTM/GRU (기울기 소실 완화)
=====================================================
"""
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# IMDB 감정분석 스타일 (텍스트 이진분류)
lstm_model = Sequential([
    layers.Embedding(input_dim=6000, output_dim=100),
    layers.LSTM(16),
    layers.Dense(1, activation="sigmoid"),
])
lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# GRU 버전 (같은 구조, 더 가벼움)
gru_model = Sequential([
    layers.Embedding(input_dim=6000, output_dim=100),
    layers.GRU(16),
    layers.Dense(1, activation="sigmoid"),
])
gru_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
"""
=====================================================
RNN 실무 예제 4 — Encoder-Decoder (시퀀스 → 시퀀스)
=====================================================
"""
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers

class EncoderDecoder(Model):
    def __init__(self, hidden_dim, num_classes):
        super().__init__()
        self.encoder = layers.SimpleRNN(hidden_dim, return_state=True)
        self.decoder = layers.SimpleRNN(hidden_dim, return_sequences=True)
        self.dense = layers.Dense(num_classes, activation="softmax")

    def call(self, encoder_inputs, decoder_inputs):
        # Encoder: 출력값은 버리고, 압축된 hidden state만 사용
        _, encoder_state = self.encoder(encoder_inputs)

        # Decoder: Encoder의 hidden state를 시작점으로 사용
        decoder_outputs = self.decoder(decoder_inputs, initial_state=[encoder_state])

        return self.dense(decoder_outputs)


# 사용 예시:
# model = EncoderDecoder(hidden_dim=20, num_classes=5)
# outputs = model(encoder_x, decoder_x)

In [ ]:
# lstm: RNN에 비해 LSTM에 추가된 3개의 게이트(forget/input/output)와 cell state는
# "정보를 얼마나 유지하고 얼마나 새로 받아들이고 얼마나 내보낼지"를 학습으로 조절해서
# RNN이 겪던 "오래된 정보가 사라지는 문제(기울기 소실)"를 완화하는 효과를 냄
# gru: LSTM의 게이트 개수를 줄여 단순화한 경량 버전. 원리는 같고 계산이 가벼움. 
# 시퀀스가 짧으면 SimpleRNN/GRU/LSTM 성능 차이가 크지 않고, 시퀀스가 길수록(예: window=300) 
# SimpleRNN이 확연히 나빠지고 LSTM/GRU가 우세해짐 (실습으로 확인됨)

In [ ]:
"""
=====================================================
RNN 실무 예제 5 — 다변량 입력 (여러 센서/지표 동시 사용)
- 예지보전에 가장 직접 대응: feature 여러 개 → 특정 값 1개 예측
=====================================================
"""
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam

window_size = 30
num_features = 4   # 예: 온도, 진동, 압력, 회전수 등 여러 센서

multivariate_model = Sequential([
    layers.LSTM(256, input_shape=(window_size, num_features)),  # 여러 feature를 한 시점에 동시 입력
    layers.Dense(64, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),   # 예: 특정 센서값 또는 고장 확률 하나만 예측
])
multivariate_model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")

# 핵심: input_shape=(window_size, num_features)
# 입력(raw_X)과 출력(raw_y)의 feature 개수가 다를 수 있음 
# (여러 센서를 보고 그 중 하나 또는 별도 지표 하나를 예측)